# Per-Sample Spike-In Limit-of-Detection Analysis

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import os
import pickle
import sys
import zlib
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

os.environ["NM_PIPELINE"] = "p3"
sys.path.insert(0, str(Path.cwd().parents[1]))
from viz_style import apply_style
apply_style()

import MixedEffectsModeling.config as config
from MixedEffectsModeling.core.calibration import bh_fdr_reject
from MixedEffectsModeling.core.dispersion_trend import load_trend
from MixedEffectsModeling.core.marginal_rqr import marginal_nb_rqr
from MixedEffectsModeling.core.model_engine_mixed import NormativeModelEngineMixed
from MixedEffectsModeling.core.shash import load_shash_params, shash_correct_matrix, shash_transform_to_z
from MixedEffectsModeling.validation.lobo_engine import (
    _score_model_genes, _score_pool, fold_covariates, load_full_data, selected_batches)

DETECTION_LIMIT_DIR = config.DETECTION_LIMIT_DIR
DETECTION_LIMIT_DIR.mkdir(exist_ok=True)

N_PERTURB_GENES = 500
LOG2FCS = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]
N_BOOTSTRAP = 5
N_FOLDS = 10  # insample_cv only
Q_LEVELS = [0.05, 0.10, 0.20, 0.25]

ALL_SPLIT_METHODS = ["lobo", "insample_cv"]
Z_CAP = 10.0  # 4_disease_scoring.ipynb

In [ ]:
data = load_full_data()

_meta_rows = []
for bdir in sorted(config.LOBO_MIXED_DIR.iterdir()):
    meta_path = bdir / "meta.json"
    if not meta_path.exists():
        continue
    meta = json.loads(meta_path.read_text())
    n_hc_test = int(sum(meta["test_is_hc"]))
    _meta_rows.append(dict(batch_id=meta["batch_id"], safe_dir=bdir.name, n_hc_test=n_hc_test))
lobo_batches = pd.DataFrame(_meta_rows).sort_values("n_hc_test", ascending=False).reset_index(drop=True)

BATCHES = [m["batch_id"] for m in selected_batches()]
print(lobo_batches[lobo_batches.batch_id.isin(BATCHES)])

In [ ]:
_cache = {}
name2row = {n: i for i, n in enumerate(data["names"])}
hc_meta_global = pd.read_csv(config.ZSCORES_MIXED_DIR / "hc_meta.csv")
Z_hc_global = np.load(config.ZSCORES_MIXED_DIR / "Z_hc_shash.npy")
gene_names_global = pickle.load(open(config.ZSCORES_MIXED_DIR / "gene_names.pkl", "rb"))
gene_pos_global = {g: j for j, g in enumerate(gene_names_global)}
hc_global_rows = np.array([name2row[s] for s in hc_meta_global["sample"]])

# QC: exclude HC samples with a pathologically inflated frac(|Z|>1.96)
_frac_extreme_hc = (np.abs(Z_hc_global) > 1.96).mean(axis=1)
_q1, _q3 = np.percentile(_frac_extreme_hc, [25, 75])
HC_QC_FENCE = _q3 + 1.5 * (_q3 - _q1)
HC_QC_OUTLIER = _frac_extreme_hc > HC_QC_FENCE  # aligned to hc_meta_global / Z_hc_global rows
print(f"HC QC: excluding {HC_QC_OUTLIER.sum()}/{len(HC_QC_OUTLIER)} samples "
      f"with frac(|Z|>1.96) > {HC_QC_FENCE:.3f} (Tukey fence) from insample_cv")


def load_batch(batch_id):
    if batch_id in _cache:
        return _cache[batch_id]
    safe = lobo_batches.set_index("batch_id").loc[batch_id, "safe_dir"]
    bdir = config.LOBO_MIXED_DIR / safe
    meta = json.loads((bdir / "meta.json").read_text())
    gene_names = pickle.load(open(bdir / "gene_names.pkl", "rb"))
    Z_test = np.load(bdir / "Z_test_shash.npy")
    test_is_hc = np.array(meta["test_is_hc"])

    fits = pd.read_csv(bdir / "model_fits.csv").set_index("gene")
    fits = fits[fits["ok"]]
    shash_p = pd.read_csv(bdir / "shash_params.csv").set_index("gene")
    universe = [g for g in fits.index if g in shash_p.index and g in gene_names]  # model-route only

    is_hc, batch, small = data["is_hc"], data["batch"], data["small_hc_batches"]
    tr_idx = np.where(is_hc & (batch != batch_id) & ~np.isin(batch, list(small)))[0]
    cov = fold_covariates(data, tr_idx)
    X = cov.transform(data["obs"], data["Y"])
    scaler = StandardScaler().fit(X[tr_idx])

    hc_test_names = np.array(meta["test_names"])[test_is_hc]
    hc_test_rows = np.array([name2row[n] for n in hc_test_names])

    pool = bdir / "pool_fit.json"
    out = dict(gene_names=gene_names, gene_pos={g: j for j, g in enumerate(gene_names)},
               Z_hc=Z_test[test_is_hc], universe=universe, fits=fits, shash_p=shash_p,
               cov=cov, X=X, scaler=scaler, hc_test_rows=hc_test_rows, rescore=rescore_lobo,
               test_rows=np.array([name2row[n] for n in meta["test_names"]]), hc_pos=np.where(test_is_hc)[0],
               fits_rt=pd.read_csv(bdir / "model_fits.csv", float_precision="round_trip").set_index("gene"),
               shash_rt=pd.read_csv(bdir / "shash_params.csv", float_precision="round_trip").set_index("gene"),
               pool_fit=json.loads(pool.read_text()) if pool.exists() else None,
               alpha_fn=load_trend(config.ENGINE_MIXED_DIR / "dispersion_trend.json"))
    _cache[batch_id] = out
    return out


def mu_alpha_tau2(b, genes, X):
    Xa = np.column_stack([np.ones(len(X)), b["scaler"].transform(X)])
    mu, alpha, tau2 = {}, {}, {}
    fits = b["fits"]
    mu_cols = [c for c in fits.columns if c.startswith("mu_coef_")]
    disp_cols = [c for c in fits.columns if c.startswith("disp_coef_")]
    for g in genes:
        row = fits.loc[g]
        mu_coef = row[mu_cols].values.astype(float)
        disp_coef = row[disp_cols].values.astype(float)
        mu[g] = np.clip(np.exp(Xa @ np.nan_to_num(mu_coef, nan=0.0)), 1e-6, 1e8)
        alpha[g] = (np.exp(-Xa @ np.nan_to_num(disp_coef, nan=0.0)) if not np.all(np.isnan(disp_coef))
                    else np.full(len(X), float(row["trend_alpha"])))
        tau2[g] = float(row["tau2"])
    return mu, alpha, tau2


def perturb_genes_fold(b, genes, sample_rows, log2fc, seed):
    """Z of the injected genes and the injected counts (all genes) of sample_rows. The injection
    reference mu uses the unperturbed covariates; the injected sample is scored with its
    count-derived covariates recomputed from the injected counts (core/covariates.py)."""
    mu, alpha, tau2 = mu_alpha_tau2(b, genes, b["X"][sample_rows])
    Y0 = data["Y"][sample_rows]
    Y1 = Y0.copy()
    for g in genes:
        c = data["gene_col"][g]
        Y1[:, c] = np.round(np.maximum(Y0[:, c] + mu[g] * (2 ** log2fc - 1), 0))
    mu, alpha, tau2 = mu_alpha_tau2(b, genes, b["cov"].transform(data["obs"][sample_rows], Y1, Y0))
    shash_p = b["shash_p"]
    z_of = {}
    for g in genes:
        z_raw = marginal_nb_rqr(Y1[:, data["gene_col"][g]], mu[g], alpha[g], tau2[g], seed=seed + zlib.crc32(g.encode()) % 9973)
        srow = shash_p.loc[g]
        z = shash_transform_to_z(z_raw, srow.xi, srow.eta, srow.eps, srow.delta) if srow.ok else z_raw
        z_of[g] = np.clip(z, -50, 50)
    return z_of, Y1


def rescore_lobo(b, idx_A, Y_A):
    """Z_test_shash of the held-out HC idx_A as lobo_engine.run_one_batch scores them, with their
    counts replaced by Y_A and covariates recomputed (every held-out sample is rescored so the RQR
    draws line up; at log2fc 0 this reproduces the stored Z exactly)."""
    Y0 = data["Y"][b["test_rows"]]
    Y = Y0.copy()
    Y[b["hc_pos"][idx_A]] = Y_A
    Xs = b["scaler"].transform(b["cov"].transform(data["obs"][b["test_rows"]], Y, Y0))
    zdict, _ = _score_model_genes(b["fits_rt"], b["fits_rt"].index.tolist(), Xs, Y, data["gene_col"], b["alpha_fn"],
                                  report=False)
    if b["pool_fit"] is not None and b["pool_fit"]["ok"]:
        zdict.update(_score_pool(b["pool_fit"], list(b["pool_fit"]["gene"]), Xs, Y, data["gene_col"], 42))
    sp = b["shash_rt"]
    for g, z in zdict.items():
        if g in sp.index and bool(sp.loc[g, "ok"]):
            s = sp.loc[g]
            zdict[g] = shash_transform_to_z(z.astype(np.float64), s["xi"], s["eta"], s["eps"], s["delta"]).astype(np.float32)
    return np.column_stack([zdict[g] for g in b["gene_names"]])[b["hc_pos"][idx_A]]


def rescore_insample(b, idx_A, Y_A):
    """Z_hc_shash rows idx_A as 4_disease_scoring.ipynb scores them, with their counts replaced by Y_A
    and covariates recomputed (all HC rescored so the RQR draws line up)."""
    Y0 = data["Y"][b["hc_test_rows"]]
    Y = Y0.copy()
    Y[idx_A] = Y_A
    X = b["cov"].transform(data["obs"][b["hc_test_rows"]], Y, Y0)
    Z = np.clip(b["engine"].score(X, Y[:, b["cols"]], gene_names=b["gene_names"]), -Z_CAP, Z_CAP)
    Zs = np.clip(shash_correct_matrix(np.nan_to_num(Z, nan=0.0, posinf=10.0, neginf=-10.0), b["gene_names"], b["shash_full"]),
                 -Z_CAP, Z_CAP)
    Zs[~np.isfinite(Z)] = np.nan
    return Zs[idx_A]

In [ ]:
def iter_lobo_folds():
    for batch_id in BATCHES:
        b = load_batch(batch_id)
        n_hc = len(b["hc_test_rows"])
        for boot in range(N_BOOTSTRAP):
            rng = np.random.default_rng(1000 * boot + zlib.crc32(batch_id.encode()) % 9973)
            perm = rng.permutation(n_hc)
            idx_A = perm[: n_hc // 2]
            rows_A = b["hc_test_rows"][idx_A]
            yield dict(group_id=batch_id, b=b, rows_A=rows_A, idx_A=idx_A, boot=boot)


_insample_cache = None


def load_insample_model():
    global _insample_cache
    if _insample_cache is not None:
        return _insample_cache

    genes = pickle.load(open(config.ENGINE_MIXED_DIR / "genes.pkl", "rb"))
    summary = pd.read_csv(config.ENGINE_MIXED_DIR / "training_summary.csv").set_index("gene")
    scaler_ins = pickle.load(open(config.ENGINE_MIXED_DIR / "scaler.pkl", "rb"))

    universe = [g for g in summary.index[summary["ok"].fillna(False)] if g in genes and genes[g].mu_coef is not None]
    recs = [genes[g] for g in universe]
    mu_coef_mat = np.nan_to_num(np.stack([r.mu_coef for r in recs]).astype(float), nan=0.0)
    disp_coef_mat = np.stack([r.disp_coef for r in recs]).astype(float)
    has_disp = ~np.all(np.isnan(disp_coef_mat), axis=1)
    disp_coef_mat = np.nan_to_num(disp_coef_mat, nan=0.0)
    tau2_vec = np.array([r.tau2 for r in recs], dtype=float)
    trend_alpha_vec = np.array([r.trend_alpha for r in recs], dtype=float)

    su = summary.loc[universe]
    ok_arr = su["cv_shash_ok"].fillna(False).infer_objects(copy=False).values.astype(bool)
    xi_arr = np.where(ok_arr, su["cv_shash_xi"].fillna(0.0).values, 0.0)
    eta_arr = np.where(ok_arr, su["cv_shash_eta"].fillna(1.0).values, 1.0)
    eps_arr = np.where(ok_arr, su["cv_shash_eps"].fillna(0.0).values, 0.0)
    delta_arr = np.where(ok_arr, su["cv_shash_delta"].fillna(1.0).values, 1.0)

    mu_cols = [f"mu_coef_{i}" for i in range(mu_coef_mat.shape[1])]
    disp_cols = [f"disp_coef_{i}" for i in range(disp_coef_mat.shape[1])]
    fits = pd.DataFrame(mu_coef_mat, columns=mu_cols, index=universe)
    for i, c in enumerate(disp_cols):
        fits[c] = np.where(has_disp, disp_coef_mat[:, i], np.nan)
    fits["trend_alpha"] = trend_alpha_vec
    fits["tau2"] = tau2_vec
    shash_p = pd.DataFrame(dict(xi=xi_arr, eta=eta_arr, eps=eps_arr, delta=delta_arr, ok=ok_arr), index=universe)

    engine = NormativeModelEngineMixed.load(config.ENGINE_MIXED_DIR)
    out = dict(gene_names=gene_names_global, gene_pos=gene_pos_global, Z_hc=Z_hc_global,
               universe=universe, fits=fits, shash_p=shash_p, scaler=scaler_ins,
               hc_test_rows=hc_global_rows, rescore=rescore_insample,
               engine=engine, cov=engine.covariates, X=engine.covariates.transform(data["obs"], data["Y"]),
               cols=[data["gene_col"][g] for g in gene_names_global],
               shash_full=load_shash_params(config.ENGINE_MIXED_DIR / "training_summary.csv"))
    _insample_cache = out
    return out


def iter_insample_cv_folds():
    b = load_insample_model()
    keep = np.where(~HC_QC_OUTLIER)[0]  # drop pathologically miscalibrated HC before CV-splitting
    n = len(keep)
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
    for fold, (_, idx_A_k) in enumerate(kf.split(np.arange(n))):
        idx_A = keep[idx_A_k]
        rows_A = b["hc_test_rows"][idx_A]
        yield dict(group_id=f"insample_fold{fold}", b=b, rows_A=rows_A, idx_A=idx_A, boot=fold)


SPLIT_PROVIDERS = {"lobo": iter_lobo_folds, "insample_cv": iter_insample_cv_folds}

In [ ]:
def run_per_sample(split_method):
    cache_path = DETECTION_LIMIT_DIR / f"sweep_persample_fold__{split_method}.csv"
    if cache_path.exists():
        return pd.read_csv(cache_path)

    rows = []
    for fold in SPLIT_PROVIDERS[split_method]():
        group_id, b = fold["group_id"], fold["b"]
        rows_A, idx_A, boot = fold["rows_A"], fold["idx_A"], fold["boot"]
        universe = b["universe"]

        for log2fc in LOG2FCS:
            rng = np.random.default_rng(int(1e6 * log2fc) + boot + zlib.crc32(group_id.encode()) % 9973)
            pert_genes = list(rng.choice(universe, min(N_PERTURB_GENES, len(universe)), replace=False))
            z_of, Y_A = perturb_genes_fold(b, pert_genes, rows_A, log2fc, seed=int(1e6 * log2fc) + boot)

            # the recomputed covariates move every gene, so all genes are rescored
            Z_A = b["rescore"](b, idx_A, Y_A)
            if log2fc == 0:
                assert ((Z_A == b["Z_hc"][idx_A]) | (np.isnan(Z_A) & np.isnan(b["Z_hc"][idx_A]))).all(), group_id
            labels = np.zeros(len(b["gene_names"]), dtype=bool)
            for g in pert_genes:
                j = b["gene_pos"][g]
                Z_A[:, j] = z_of[g]
                labels[j] = True

            finite = np.isfinite(Z_A)
            p = np.where(finite, 2 * norm.sf(np.abs(np.where(finite, Z_A, 0.0))), np.nan)

            for q in Q_LEVELS:
                tp = fp = fn_ = tn = 0
                for i in range(Z_A.shape[0]):
                    fin_i = finite[i]
                    reject_i = np.zeros(Z_A.shape[1], dtype=bool)
                    reject_i[fin_i] = bh_fdr_reject(p[i, fin_i], q=q)
                    tp += int((reject_i & labels).sum())
                    fp += int((reject_i & ~labels).sum())
                    fn_ += int((~reject_i & labels).sum())
                    tn += int((~reject_i & ~labels).sum())
                rows.append(dict(batch=group_id, boot=boot, log2fc=log2fc, q=q,
                                 n_samples=int(Z_A.shape[0]), n_universe=int(Z_A.shape[1]),
                                 n_perturbed=len(pert_genes), tp=tp, fp=fp, fn=fn_, tn=tn))
        print(split_method, group_id, "per-sample done", flush=True)

    df = pd.DataFrame(rows)
    df.to_csv(cache_path, index=False)
    return df


def run_per_sample_all_splits():
    parts = []
    for split_method in ALL_SPLIT_METHODS:
        df = run_per_sample(split_method)
        df["split"] = split_method
        parts.append(df)
    return pd.concat(parts, ignore_index=True)


sweep_persample = run_per_sample_all_splits()
sweep_persample

In [ ]:
counts_ps = sweep_persample.groupby(["split", "q", "log2fc"])[["tp", "fp", "fn", "tn"]].sum()
tp, fp, fn, tn = counts_ps["tp"], counts_ps["fp"], counts_ps["fn"], counts_ps["tn"]
summary_ps = pd.DataFrame(dict(
    TP=tp, FP=fp, FN=fn, TN=tn,
    Precision=tp / (tp + fp), Sensitivity=tp / (tp + fn), Specificity=tn / (tn + fp),
    Accuracy=(tp + tn) / (tp + fp + fn + tn), FDR=fp / (tp + fp),
    F1=2 * tp / (2 * tp + fp + fn),
)).round(3)

for sm in ALL_SPLIT_METHODS:
    print(f"per-sample (no aggregation) -- split method: {sm}")
    display(summary_ps.loc[sm])

null_calib_ps = sweep_persample[sweep_persample.log2fc == 0].groupby(["split", "q"])["fp"].agg(["mean", "max"])
print("null (log2fc=0) per-sample FP calibration:")
display(null_calib_ps)

In [ ]:
import matplotlib.pyplot as plt

_PLOT_SPLITS = ["insample_cv", "lobo"]
_METRICS = ["Sensitivity", "Precision", "FDR", "Specificity"]
_LINESTYLES = {"Sensitivity": "-", "Precision": "--", "FDR": ":", "Specificity": "-."}
_MARKERS = {"Sensitivity": "o", "Precision": "s", "FDR": "^", "Specificity": "D"}

# grayscale gradient from black -- edit this list to restyle
PALETTE = {"Sensitivity": "#000000", "Precision": "#1916BB", "FDR": "#5250EB", "Specificity": "#9896F3"}

_plot_df = summary_ps.reset_index()
_plot_df = _plot_df[(_plot_df.split.isin(_PLOT_SPLITS)) & (_plot_df.log2fc > 0)]
_qs = sorted(_plot_df.q.unique())

fig, axes = plt.subplots(len(_qs), len(_PLOT_SPLITS), figsize=(5 * len(_PLOT_SPLITS), 2.5 * len(_qs)),
                         sharex=False, sharey=True, squeeze=False)
for i, q in enumerate(_qs):
    for j, sm in enumerate(_PLOT_SPLITS):
        ax = axes[i, j]
        sub = _plot_df[(_plot_df.q == q) & (_plot_df.split == sm)].sort_values("log2fc")
        for metric in _METRICS:
            ax.plot(sub.log2fc, sub[metric], color=PALETTE[metric], linestyle=_LINESTYLES[metric],
                    marker=_MARKERS[metric], label=metric)
        ax.axhline(q, color='red', linestyle=":", linewidth=1,
                  label="FDR target (q)" if (i, j) == (0, 0) else None)
        ax.set_ylim(-0.00, 1.05)
        if i == 0:
            if sm == "insample_cv":
                ax.set_title("Insample CV")
            elif sm == "lobo":
                ax.set_title("Unseen Batch (LOBO)")
        if j == 0:
            ax.set_ylabel(f"q={q}")
        if i == len(_qs) - 1:
            ax.set_xlabel("log2fc")
        ax.grid(alpha=0.2)
axes[2, -1].legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False)
plt.tight_layout()
plt.savefig(DETECTION_LIMIT_DIR / "per_sample_sweep.png", dpi=300)